# Augmentation sélective du corpus stable

Ce test conserve intégralement les 4 459 chunks de référence. Il ajoute seulement les chunks structurés apportant un vocabulaire réellement nouveau sur des pages où Trafilatura avait extrait moins de 1 000 caractères alors que le contenu principal en contient au moins 3 000. Tout est indexé dans un dossier isolé et les index réels sont protégés par empreinte.


In [1]:
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
from collections import defaultdict
import frontmatter
import hashlib, json, logging, re, shutil, sys, unicodedata
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import load_config
from chunker import Chunk
from embeddings import EmbeddingModel
from evaluation import load_golden_set, RetrievalEvaluator
from retriever import HybridRetriever, Reranker
from vectorstore import VectorStoreManager

logging.basicConfig(level=logging.INFO)
cfg = load_config(ROOT / 'config.yaml')


In [2]:
def tree_fingerprint(root):
    digest = hashlib.sha256()
    for path in sorted(root.rglob('*')):
        if path.is_file():
            digest.update(str(path.relative_to(root)).encode('utf-8'))
            with open(path, 'rb') as handle:
                for block in iter(lambda: handle.read(1024 * 1024), b''):
                    digest.update(block)
    return digest.hexdigest()

def token_set(text):
    normalized = unicodedata.normalize('NFKD', text).lower()
    return set(re.findall(r'(?u)\b\w{3,}\b', normalized))

def is_content_surface(url):
    rejected = ('/logout', 'fusion_tb_category', '/slide-page/', '/user-profile/', 'alumni@')
    return not any(part in url.lower() for part in rejected)

experiment_root = ROOT / 'data' / 'experiments'
cleaned_dirs = list(experiment_root.glob('cleaned_candidate_*'))
candidate_chunk_files = list(experiment_root.glob('chunks_candidate_*.jsonl'))
assert cleaned_dirs and candidate_chunk_files, 'Exécutez d’abord les notebooks 05 et 06.'
cleaned_dir = max(cleaned_dirs, key=lambda path: path.stat().st_mtime)
candidate_chunks_path = max(candidate_chunk_files, key=lambda path: path.stat().st_mtime)
baseline_chunks_path = ROOT / 'data' / 'chunks' / 'chunks.jsonl'

selected_urls = {}
for path in cleaned_dir.glob('*.md'):
    post = frontmatter.load(path)
    meta = post.metadata
    diagnostics = meta.get('extraction_diagnostics') or {}
    url = str(meta.get('url') or '')
    trafilatura_chars = int(diagnostics.get('trafilatura_chars') or 0)
    structured_chars = int(diagnostics.get('structured_main_chars') or 0)
    if (
        meta.get('extraction_method') == 'structured_main_richer'
        and is_content_surface(url)
        and trafilatura_chars <= 1000
        and structured_chars >= 3000
        and structured_chars - trafilatura_chars >= 2500
        and structured_chars / max(trafilatura_chars, 1) >= 4.0
    ):
        selected_urls[url] = {
            'trafilatura_chars': trafilatura_chars,
            'structured_chars': structured_chars,
        }

print(f'Pages enrichissables retenues : {len(selected_urls)}')
display(pd.DataFrame.from_dict(selected_urls, orient='index').sort_values('structured_chars', ascending=False).head(20))


Pages enrichissables retenues : 46


,trafilatura_chars,structured_chars
https://ensea.ed.ci/faq/,704,17438
https://ensea.ed.ci/formation-initiale/master-statistiques-agricoles-2025/,845,11364
https://ensea.ed.ci/initial-education/master-in-agricultural-statistics/,881,10849
https://ensea.ed.ci/initial-education/master-in-agricultural-statistics/?lang=en,881,10849
https://ensea.ed.ci/formation-initiale/ise/,685,9620
https://ensea.ed.ci/notre-equipe/,537,9572
https://ensea.ed.ci/our-team/,674,8869
https://ensea.ed.ci/initial-education/statistical-economist-engineer/,618,8498
https://ensea.ed.ci/initial-education/statistical-economist-engineer/?lang=en,618,8498
https://ensea.ed.ci/etudes-et-recherche/publications/,15,8037


In [3]:
def load_rows(path):
    with open(path, encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

baseline_rows = load_rows(baseline_chunks_path)
candidate_rows = load_rows(candidate_chunks_path)
baseline_ids = {row['chunk_id'] for row in baseline_rows}
baseline_tokens_by_url = defaultdict(set)
for row in baseline_rows:
    baseline_tokens_by_url[row['url']].update(token_set(row['text']))

supplemental_rows = []
novelty_report = []
for row in candidate_rows:
    if row['url'] not in selected_urls or row['chunk_id'] in baseline_ids:
        continue
    tokens = token_set(row['text'])
    new_tokens = tokens - baseline_tokens_by_url[row['url']]
    novelty_ratio = len(new_tokens) / max(len(tokens), 1)
    if len(new_tokens) >= 20 and novelty_ratio >= 0.20:
        supplemental_rows.append(row)
        novelty_report.append({
            'url': row['url'],
            'title': row['title'],
            'new_tokens': len(new_tokens),
            'novelty_ratio': novelty_ratio,
        })

augmented_rows = baseline_rows + supplemental_rows
assert len({row['chunk_id'] for row in augmented_rows}) == len(augmented_rows)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
augmented_chunks_path = experiment_root / f'chunks_augmented_{stamp}.jsonl'
augmented_chunks_path.write_text(
    ''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in augmented_rows),
    encoding='utf-8',
)
print(f'Chunks stables conservés : {len(baseline_rows)}')
print(f'Chunks supplémentaires : {len(supplemental_rows)}')
print(f'Total augmenté : {len(augmented_rows)}')
display(pd.DataFrame(novelty_report).groupby('url').agg(chunks=('url', 'size'), nouveaux_tokens=('new_tokens', 'sum')).sort_values('nouveaux_tokens', ascending=False).head(20))


Chunks stables conservés : 4459
Chunks supplémentaires : 221
Total augmenté : 4680


,chunks,nouveaux_tokens
url,,
https://ensea.ed.ci/notre-equipe/,8,730
https://ensea.ed.ci/formation-initiale/ise/,8,679
https://ensea.ed.ci/formation-initiale/master-statistiques-agricoles-2025/,8,679
https://ensea.ed.ci/etudes-et-recherche/publications/,7,678
https://ensea.ed.ci/studies-and-research/publications/,7,678
https://ensea.ed.ci/initial-education/master-in-agricultural-statistics/,8,670
https://ensea.ed.ci/initial-education/master-in-agricultural-statistics/?lang=en,8,670
https://ensea.ed.ci/our-team/,7,623
https://ensea.ed.ci/etudes-et-recherche/etudes/,7,565


In [4]:
real_lancedb = cfg.vectorstore.uri_path
real_bm25 = cfg.vectorstore.bm25_index_full_path
real_lancedb_hash_before = tree_fingerprint(real_lancedb)
real_bm25_hash_before = hashlib.sha256(real_bm25.read_bytes()).hexdigest()

augmented_index_dir = experiment_root / f'index_augmented_{stamp}'
augmented_index_dir.mkdir(parents=True, exist_ok=False)
augmented_lancedb = augmented_index_dir / 'lancedb'
augmented_bm25 = augmented_index_dir / 'bm25_index.pkl'
shutil.copytree(real_lancedb, augmented_lancedb)
shutil.copy2(real_bm25, augmented_bm25)

augmented_cfg = cfg.model_copy(deep=True)
augmented_cfg.vectorstore.uri = str(augmented_lancedb)
augmented_cfg.vectorstore.bm25_index_path = str(augmented_bm25)
augmented_cfg.observability.enabled = False
augmented_chunks = [Chunk(**row) for row in augmented_rows]
embedding_model = EmbeddingModel(augmented_cfg.embeddings)
augmented_store = VectorStoreManager(augmented_cfg)
augmented_store.load_sparse()
index_report = augmented_store.sync_incremental(augmented_chunks, embedding_model)
display(pd.Series(index_report.__dict__, name='valeur').to_frame())


,valeur
changed_urls,46.00000
embedded_chunks,264.00000
reused_chunks,4416.00000
removed_chunks,43.00000
total_chunks,4680.00000
embedding_seconds,5.43692


In [5]:
reranker = Reranker(augmented_cfg.reranker.model_name, augmented_cfg.reranker.device)
retriever = HybridRetriever(augmented_cfg, embedding_model, augmented_store, reranker)
pipeline = SimpleNamespace(retriever=retriever)
all_examples = load_golden_set(augmented_cfg.evaluation.expanded_golden_set_full_path)
_ = retriever.retrieve('Quelles formations propose l’ENSEA ?')
print('Retriever augmenté initialisé et échauffé.')


INFO:vectorstore:Cache RAM des vecteurs chargé : 4680 vecteurs (18.3 Mo).
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
INFO:retriever:Retrieval 6.745s | query=0.038s dense=0.031s bm25=0.004s fusion=0.000s vectors+MMR=1.210s rerank=5.462s | 5 résultat(s)


Retriever augmenté initialisé et échauffé.


In [6]:
baselines = {
    'regression': {'mrr': 0.7183333333333334, 'hit_rate@5': 0.95, 'mean_latency_seconds': 0.3943393000001379},
    'development': {'mrr': 0.7875, 'hit_rate@5': 0.85, 'mean_latency_seconds': 0.3963170174996776},
}
benchmark_results = {}
benchmark_details = {}
comparisons = []
for split in ('regression', 'development'):
    examples = [example for example in all_examples if example.split == split]
    evaluator = RetrievalEvaluator(augmented_cfg.evaluation)
    summary = evaluator.evaluate(pipeline, examples)
    benchmark_results[split] = summary
    benchmark_details[split] = evaluator.last_details
    print(f'\n{split.upper()}')
    display(pd.Series(summary, name='valeur').to_frame())
    details = pd.DataFrame(evaluator.last_details)
    misses = details[details['should_answer'] & details['first_relevant_rank'].isna()]
    print(f'Échecs stricts : {len(misses)}/{len(details)}')
    display(misses[['id', 'question', 'category', 'top_title', 'top_url', 'relevant_urls']])
    for metric in ('mrr', 'hit_rate@5', 'mean_latency_seconds'):
        comparisons.append({
            'split': split,
            'metric': metric,
            'référence': baselines[split][metric],
            'augmenté': summary[metric],
            'écart': summary[metric] - baselines[split][metric],
        })

comparison_df = pd.DataFrame(comparisons)
display(comparison_df)
quality_ok = all(
    benchmark_results[split]['mrr'] >= baselines[split]['mrr'] - 1e-9
    and benchmark_results[split]['hit_rate@5'] >= baselines[split]['hit_rate@5'] - 1e-9
    for split in baselines
)
latency_ok = all(
    benchmark_results[split]['mean_latency_seconds'] <= baselines[split]['mean_latency_seconds'] * 1.15
    for split in baselines
)
has_gain = any(
    benchmark_results[split]['mrr'] > baselines[split]['mrr'] + 1e-9
    or benchmark_results[split]['hit_rate@5'] > baselines[split]['hit_rate@5'] + 1e-9
    for split in baselines
)
promotion_recommended = quality_ok and latency_ok and has_gain
print({'quality_ok': quality_ok, 'latency_ok': latency_ok, 'has_gain': has_gain, 'promotion_recommended': promotion_recommended})


INFO:retriever:Retrieval 0.453s | query=0.068s dense=0.021s bm25=0.005s fusion=0.000s vectors+MMR=0.003s rerank=0.357s | 5 résultat(s)
INFO:retriever:Retrieval 0.440s | query=0.046s dense=0.023s bm25=0.005s fusion=0.000s vectors+MMR=0.002s rerank=0.364s | 5 résultat(s)
INFO:retriever:Retrieval 0.451s | query=0.063s dense=0.018s bm25=0.005s fusion=0.000s vectors+MMR=0.001s rerank=0.363s | 5 résultat(s)
INFO:retriever:Retrieval 0.424s | query=0.029s dense=0.021s bm25=0.003s fusion=0.000s vectors+MMR=0.003s rerank=0.368s | 5 résultat(s)
INFO:retriever:Retrieval 0.405s | query=0.033s dense=0.020s bm25=0.005s fusion=0.000s vectors+MMR=0.003s rerank=0.343s | 5 résultat(s)
INFO:retriever:Retrieval 0.432s | query=0.042s dense=0.020s bm25=0.005s fusion=0.000s vectors+MMR=0.002s rerank=0.363s | 5 résultat(s)
INFO:retriever:Retrieval 0.390s | query=0.031s dense=0.020s bm25=0.003s fusion=0.000s vectors+MMR=0.002s rerank=0.334s | 5 résultat(s)
INFO:retriever:Retrieval 0.401s | query=0.040s dense=0.


REGRESSION


,valeur
questions,20.000000
answerable_questions,20.000000
unanswerable_questions,0.000000
mrr,0.735000
mean_latency_seconds,0.417110
p50_latency_seconds,0.416360
p95_latency_seconds,0.453024
max_latency_seconds,0.460684
recall@3,0.685417
precision@3,0.333333


Échecs stricts : 1/20


,id,question,category,top_title,top_url,relevant_urls
8,reg-009,Quels sont les laboratoires de recherche assoc...,recherche,IRD : Institut de Recherche pour le Développem...,https://ensea.ed.ci/partenaires/institut-de-re...,[https://ensea.ed.ci/partenaires/laboratoire-d...


...

,valeur
questions,40.000000
answerable_questions,40.000000
unanswerable_questions,0.000000
mrr,0.758333
mean_latency_seconds,0.405648
p50_latency_seconds,0.420874
p95_latency_seconds,0.436011
max_latency_seconds,0.443999
recall@3,0.641667
precision@3,0.333333


Échecs stricts : 8/40


,id,question,category,top_title,top_url,relevant_urls
5,ext-006,Que propose le Master en Management de l'ENSEA ?,formation_initiale,À propos - ENSEA,https://ensea360.ensea.ed.ci/a-propos/,[https://ensea.ed.ci/formation-initiale/master...
7,ext-008,Que couvre la formation continue de l'ENSEA ?,formation_continue,Formation Continue / L’ENSEA s’engage à renfor...,https://ensea.ed.ci/formation-continue-lensea-...,"[https://ensea.ed.ci/formation-continue/, http..."
9,ext-010,Comment est organisée la formation doctorale d...,doctorat,Formation Doctorale: Séminaire de Formation Do...,https://ensea.ed.ci/formation-doctorale-semina...,"[https://ensea.ed.ci/formation-doctorale/, htt..."
13,ext-014,Où consulter la liste des doctorants de l'ENSEA ?,doctorat,Admission Doctorat - ENSEA,https://ensea360.ensea.ed.ci/formation-doctora...,[https://ensea.ed.ci/formation-doctorale/docto...
14,ext-015,L'ENSEA organise-t-elle des séminaires de rech...,recherche,765106d61b6f9473,https://ensea.ed.ci/wp-content/uploads/2024/06...,[https://ensea360.ensea.ed.ci/formation-contin...
26,ext-027,Où sont indiqués les coûts des formations init...,financement,,,[https://ensea.ed.ci/formation-initiale/couts/]
27,ext-028,Où puis-je consulter les tarifs de l'ENSEA ?,financement,Client entreprise - ENSEA,https://ensea360.ensea.ed.ci/formation-continu...,[https://ensea.ed.ci/faq-items/quels-sont-les-...
29,ext-030,Quel niveau faut-il pour intégrer la filière AS ?,admission,Filière des Analystes Statisticiens (AS) - ENSEA,https://ensea.ed.ci/formation-initiale/as/,[https://ensea.ed.ci/faq-items/quel-est-le-niv...


,split,metric,référence,augmenté,écart
0,regression,mrr,0.718333,0.735000,0.016667
1,regression,hit_rate@5,0.950000,0.950000,0.000000
2,regression,mean_latency_seconds,0.394339,0.417110,0.022770
3,development,mrr,0.787500,0.758333,-0.029167
4,development,hit_rate@5,0.850000,0.800000,-0.050000
5,development,mean_latency_seconds,0.396317,0.405648,0.009331


{'quality_ok': False, 'latency_ok': True, 'has_gain': True, 'promotion_recommended': False}


In [7]:
result_path = augmented_index_dir / 'benchmark_results.json'
result_path.write_text(json.dumps({
    'selection': {
        'selected_urls': selected_urls,
        'baseline_chunks': len(baseline_rows),
        'supplemental_chunks': len(supplemental_rows),
        'total_chunks': len(augmented_rows),
    },
    'index_report': index_report.__dict__,
    'benchmarks': benchmark_results,
    'details': benchmark_details,
    'promotion_recommended': promotion_recommended,
}, ensure_ascii=False, indent=2), encoding='utf-8')

assert augmented_store.dense.count() == len(augmented_chunks)
assert tree_fingerprint(real_lancedb) == real_lancedb_hash_before
assert hashlib.sha256(real_bm25.read_bytes()).hexdigest() == real_bm25_hash_before
print('Test terminé ; LanceDB et BM25 réels sont strictement intacts.')
print(f'Promotion recommandée : {promotion_recommended}')
print(f'Résultats : {result_path}')


Test terminé ; LanceDB et BM25 réels sont strictement intacts.
Promotion recommandée : False
Résultats : <PROJECT_ROOT>\data\experiments\index_augmented_20260803_131920\benchmark_results.json
